# Figuras

Genera las figuras 3.1 y 4.1 a 4.5 y la Tabla 4.5. La Figura 3.2 se genera en `02_EDA`.

## Rutas

Las rutas son relativas a la carpeta `notebooks/`. Los datos no forman parte del repositorio (ver README).

In [ ]:
# Comprobación de rutas
import os
PROY = os.path.abspath('..')
print("carpeta del notebook :", os.getcwd())
print("raiz del proyecto    :", PROY)
for _sub in ("Topic/Dataframes", "Topic/Resultados", "csv_parques"):
    _p = os.path.join(PROY, _sub)
    print(f"  {'OK ' if os.path.isdir(_p) else 'FALTA'}  {_sub}")
print("\nSi algo dice FALTA, el kernel no arranco en la carpeta del notebook,\n"
      "o los datos no estan donde se los espera (ver README).")

### Figura 3.1

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

BD = ".."
df = pd.read_pickle(f"{BD}/df_generic_cleaned.pkl")   # mismo origen que 02_EDA

# Etiquetas idénticas a las del resto de las figuras
nombres = {
    'botanico':                       'Jardín Botánico',
    'parque_prado':                   'Parque del Prado',
    'parque_rodo':                    'Parque Rodó',
    'rosedal-del_prado':              'Rosedal del Prado',
    'villa_dolores':                  'Villa Dolores',
    'jose_batlle_y_ordoniez':         'José Batlle y Ordóñez',
    'espacio_modelo':                 'Espacio Modelo',
    'idea_vilarinio':                 'Idea Vilariño',
    'museo_y-parque_fernando_garcia': 'Museo y P. Fernando García',
    'punta_yeguas':                   'Punta Yeguas',
}

# Chequeo: todo identificador del df tiene nombre asignado
faltan = set(df['nombre_parque'].unique()) - set(nombres)
assert not faltan, f"Identificadores sin nombre: {faltan}"

def pc(v):
    return f"{v:.2f}".replace(".", ",")

tabla = (
    df.groupby('nombre_parque')
      .agg(cantidad_comentarios=('text', 'count'))
      .reset_index()
      .sort_values('cantidad_comentarios', ascending=False)
)
total = tabla['cantidad_comentarios'].sum()
tabla['peso_relativo_%'] = tabla['cantidad_comentarios'] / total * 100
tabla['parque'] = tabla['nombre_parque'].map(nombres)

print(f"total = {total:,}   (esperado 27,320)")
print(tabla[['parque', 'cantidad_comentarios', 'peso_relativo_%']]
      .assign(**{'peso_relativo_%': tabla['peso_relativo_%'].map(pc)})
      .to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 6))
barras = ax.bar(tabla['parque'], tabla['cantidad_comentarios'],
                color="steelblue", edgecolor="white")

for barra, (_, fila) in zip(barras, tabla.iterrows()):
    ax.text(barra.get_x() + barra.get_width() / 2, barra.get_height() + 0.3,
            f"{int(fila['cantidad_comentarios']):,}".replace(",", ".") +
            f"\n({pc(fila['peso_relativo_%'])}%)",
            ha="center", va="bottom", fontsize=9)

ax.set_title("Cantidad de comentarios por parque", fontsize=14, fontweight="bold")
ax.set_xlabel("Parque")
ax.set_ylabel("Cantidad de comentarios")
ax.set_ylim(0, 7600)
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig(f"{BD}/EDA_comentarios_por_parque_v2.png", dpi=300)
plt.show()

### Figura 4.1

In [ ]:
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from sklearn.feature_extraction.text import CountVectorizer

def pc(v, d=2):
    """Formatea con coma decimal, como el resto del documento."""
    return f"{v:.{d}f}".replace(".", ",")

BD     = ".."
DFDIR  = f"{BD}/Topic/Dataframes"
RES    = f"{BD}/Topic/Resultados/Topico"
SALIDA = f"{RES}/figura_4_1_destino_del_residuo_v2.png"

TITULO = ("Destino de las reseñas sin términos en el vocabulario final de LDA y TF-IDF, "
          "por configuración")
EJE_X  = "Porcentaje del conjunto seguido en cada configuración"

UMBRAL_AGREGACION = 2.0   # % por debajo del cual un destino se agrega al "resto"

# --- 1. Máscara de reseñas sin representación léxica ---
prep = pickle.load(open(f"{DFDIR}/prep_lemmatizado.pkl", "rb"))
cv = CountVectorizer(stop_words=prep['final_stopwords'], min_df=10, max_df=0.6, ngram_range=(1, 2))
X = cv.fit_transform(prep['corpus'])
vac = np.asarray(X.getnnz(axis=1) == 0)

df_lda = pd.read_pickle(f"{DFDIR}/df_lda_final.pkl")
assert len(df_lda) == len(prep['corpus']), "desalineado corpus <-> df_lda_final"
ids_vac = set(df_lda.loc[vac, 'id'])
N_RES = len(ids_vac)

def conteo(ids, etiquetas):
    d = pd.DataFrame({'id': np.asarray(ids), 'lab': np.asarray(etiquetas)})
    return d[d['id'].isin(ids_vac)]['lab'].value_counts().to_dict()

# --- 2. Destinos. queda_asignada=False -> el documento no queda en ningún tópico ---
dt = pd.read_pickle(f"{DFDIR}/df_tfidf_final.pkl")
dw = pd.read_pickle(f"{DFDIR}/df_w2v_final.pkl")
db = pd.read_pickle(f"{DFDIR}/df_bertopic_final.pkl")
colb = next(c for c in db.columns if 'topic' in c.lower() or 'topico' in c.lower())
dl = pd.read_csv(f"{RES}/LLM_results_with_id.csv")

c_lda, c_tfidf, c_bert, c_llm = (conteo(df_lda['id'], df_lda['tema_dominante']),
                                 conteo(dt['id'], dt['cluster_tema']),
                                 conteo(db['id'], db[colb]),
                                 conteo(dl['id'], dl['topic_id']))
n_w2v_ok = sum(conteo(dw['id'], dw['cluster_w2v']).values())

destinos = {
 'LDA': [('Tópico 0 — asignado por\nla distribución a priori', c_lda.get(0, 0), True)],
 'K-Means + TF-IDF': [('Cluster 3 — vectores nulos', c_tfidf.get(3, 0), True)],
 'K-Means + Word2Vec': [('Descartadas — sin vector', N_RES - n_w2v_ok, False),
                        ('Con vector — repartidas\nentre tópicos', n_w2v_ok, True)],
 'BERTopic': [('Ruido (HDBSCAN)', c_bert.get(-1, 0), False),
              ('T0 — tópico mayoritario', c_bert.get(0, 0), True),
              ('T1 — registro evaluativo', c_bert.get(1, 0), True),
              ('T3 — registro\nevaluativo', c_bert.get(3, 0), True),
              ('Resto de los tópicos', sum(v for k, v in c_bert.items() if k not in (-1, 0, 1, 3)), True)],
 'LLM (zero-shot)': [('T5 — «Experiencia y atractivo»', c_llm.get(5, 0), True),
                     ('Resto de las categorías', sum(v for k, v in c_llm.items() if k not in (5, -1)), True),
                     ('Sin clasificar', c_llm.get(-1, 0), False)],
}

# --- 3. Agregar destinos menores ---
def agregar(items):
    base = sum(n for _, n, _ in items)
    grandes = [(e, n, c) for e, n, c in items if n / base * 100 >= UMBRAL_AGREGACION]
    chicos  = [(e, n, c) for e, n, c in items if n / base * 100 <  UMBRAL_AGREGACION and n > 0]
    if len(chicos) == 1:
        grandes.append(chicos[0])
    elif chicos:
        grandes.append((f'Otros destinos ({len(chicos)})', sum(n for _, n, _ in chicos),
                        all(c for _, _, c in chicos)))
    return grandes, base

# --- 4. Chequeos (mismos valores que la corrida original de c23) ---
print("CHEQUEOS")
ok = int(vac.sum()) == 7404
print(f"  residuo total: {int(vac.sum()):,}   (esperado 7.404)")
for arq, items in destinos.items():
    tot = sum(n for _, n, _ in items)
    esp = 478 if arq.startswith('LLM') else 7404
    print(f"  {arq:<20} suma de destinos = {tot:>6,}  (esperado {esp:,})")
    ok &= tot == esp
esperados = {('BERTopic', 'Ruido (HDBSCAN)'): 2876, ('BERTopic', 'T0 — tópico mayoritario'): 2412,
             ('BERTopic', 'T1 — registro evaluativo'): 1942, ('K-Means + Word2Vec', 'Descartadas — sin vector'): 7094,
             ('LLM (zero-shot)', 'T5 — «Experiencia y atractivo»'): 428}
for (arq, etq), esp in esperados.items():
    n = next(n for e, n, _ in destinos[arq] if e == etq)
    print(f"  {arq:<20} {etq:<34} {n:>6,}  (esperado {esp:,})")
    ok &= n == esp
assert ok, "Algún chequeo falló. NO usar la figura."
print("  todo OK\n")

print(f"Reparto de las {N_RES:,} reseñas sin términos en el vocabulario final\n")
for arq, items in destinos.items():
    grandes, base = agregar(items)
    n_asignadas = sum(n for _, n, c in items if c)
    print(f"{arq}   (base {base:,})")
    for e, n, c in sorted(grandes, key=lambda t: -t[1]):
        print(f"    {e.replace(chr(10),' '):<44} {n:>6,}  ({pc(n/base*100)} %)   "
              f"{'asignada' if c else 'no asignada'}")
    print(f"    {'→ quedan asignadas a un tópico':<44} {n_asignadas:>6,}  ({pc(n_asignadas/base*100)} %)\n")

# --- 5. Figura ---
AZUL = ['#2F4A9E', '#5877C4', '#93A9DF']     # no queda asignada a ningún tópico
OCRE = ['#C2762A', '#E0A868', '#F2D6AA']     # queda asignada a un tópico
CORTO = {'Tópico 0 — asignado por\nla distribución a priori': 'Tópico 0\nasignado por el prior',
         'Cluster 3 — vectores nulos': 'Cluster 3\nvectores nulos',
         'Descartadas — sin vector': 'Descartadas\nsin vector',
         'Con vector — repartidas\nentre tópicos': 'con vector',
         'Ruido (HDBSCAN)': 'Ruido\nHDBSCAN',
         'T0 — tópico mayoritario': 'T0\ntópico mayoritario',
         'T1 — registro evaluativo': 'T1\nregistro evaluativo',
         'T3 — registro\nevaluativo': 'T3 evaluativo',
         'Resto de los tópicos': 'resto',
         'T5 — «Experiencia y atractivo»': 'T5 — «Experiencia y atractivo»',
         'Resto de las categorías': 'resto de\nlas categorías',
         'Sin clasificar': 'sin clasificar'}
arqs = list(destinos.keys())[::-1]

ALTO_BARRA = 0.44
BORDE = ALTO_BARRA / 2

fig, ax = plt.subplots(figsize=(12, 6.8))
for i, arq in enumerate(arqs):
    grandes = sorted(agregar(destinos[arq])[0], key=lambda t: -t[1])
    base = sum(n for _, n, _ in grandes)
    izq, k_az, k_oc, lado = 0.0, 0, 0, 0
    for etq, n, asignada in grandes:
        pct = n / base * 100
        if asignada: color, texto = OCRE[min(k_oc, 2)], '#33210A'; k_oc += 1
        else:        color, texto = AZUL[min(k_az, 2)], 'white';   k_az += 1
        ax.barh(i, pct, left=izq, color=color, edgecolor='white', linewidth=1.5, height=ALTO_BARRA)
        nombre = CORTO.get(etq, etq)
        if pct >= 13:
            ax.text(izq + pct/2, i, f"{nombre}\n{pc(pct)}%", ha='center', va='center',
                    fontsize=8, color=texto, linespacing=1.5)
        else:
            dy = -(BORDE + 0.13) if lado == 0 else -(BORDE + 0.34); lado = 1 - lado
            ax.annotate(f"{nombre.replace(chr(10),' ')}  {pc(pct)}%",
                        xy=(izq + pct/2, i - BORDE), xytext=(izq + pct/2, i + dy),
                        ha='center', va='top', fontsize=7.2, color='#444444',
                        arrowprops=dict(arrowstyle='-', lw=0.7, color='#999999', shrinkA=0, shrinkB=1))
        izq += pct

ax.set_yticks(range(len(arqs))); ax.set_yticklabels(arqs, fontsize=10.5)
ax.set_xlim(0, 100); ax.set_ylim(-1.0, len(arqs) - 0.4)
ax.set_title(TITULO, fontsize=11.5, pad=14)
ax.set_xlabel(EJE_X, fontsize=9.5)
ax.spines[['top', 'right', 'left']].set_visible(False)
ax.tick_params(axis='y', length=0)
ax.legend(handles=[Patch(facecolor=OCRE[0], label='Queda asignada a un tópico'),
                   Patch(facecolor=AZUL[0], label='No queda asignada a ningún tópico')],
          loc='lower center', bbox_to_anchor=(0.5, -0.22), ncol=2, frameon=False, fontsize=8.8)
fig.tight_layout()
fig.savefig(SALIDA, dpi=500, bbox_inches='tight')
plt.show()
print(f"guardado -> {SALIDA}")

### Figura 4.2

In [ ]:
import pickle, unicodedata
import numpy as np
import pandas as pd
import spacy
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, Rectangle

BD    = ".."
DFDIR = f"{BD}/Topic/Dataframes"
OUT   = f"{BD}/Topic/Resultados/Topico"
TOP_N = 10

# =====================================================================
# 1. PALABRAS + TEXTOS DE CADA TÉCNICA (modelos finales, solo lectura)
#    ref_texts de cada ckpt se guardó en la misma celda y con el mismo
#    df que el df_*_final -> mismo orden de filas.
# =====================================================================
def cargar(ck):
    d  = pickle.load(open(f"{DFDIR}/{ck}", "rb"))
    sw = set(d['final_stopwords'])
    palabras = [[w for w in tw if w not in sw][:TOP_N] for tw in d['topic_words_raw']]
    docs = [set(t) for t in d['ref_texts']]          # unigramas y bigramas según la técnica
    return palabras, docs

lda_w, lda_docs = cargar("ckpt_lda.pkl")
tf_w,  tf_docs  = cargar("ckpt_tfidf.pkl")
w2v_w, w2v_docs = cargar("ckpt_w2v.pkl")
ber_w, ber_docs = cargar("ckpt_bertopic.pkl")

df_lda = pd.read_pickle(f"{DFDIR}/df_lda_final.pkl")
df_tf  = pd.read_pickle(f"{DFDIR}/df_tfidf_final.pkl")
df_w2v = pd.read_pickle(f"{DFDIR}/df_w2v_final.pkl")
df_ber = pd.read_pickle(f"{DFDIR}/df_bertopic_final.pkl")
base   = pd.read_pickle(f"{BD}/df_generic_cleaned.pkl")

# LLM: palabras del bloque M8 de 09_analisis_derivados; textos lematizados igual que M8
LLM_W = [
    ['descuidado', 'limpieza', 'limpio', 'sucio', 'faltar', 'cuidado', 'mantenimiento', 'abandonado', 'falta', 'mejorar'],
    ['juego', 'nino', 'juegos', 'chico', 'jugar', 'cuidado', 'plaza', 'variedad', 'actividad', 'cancha'],
    ['seguro', 'nino', 'familia', 'seguridad', 'cuidado', 'robar', 'plaza', 'lastima', 'tarde', 'iluminacion'],
    ['cuidado', 'naturaleza', 'limpio', 'verde', 'familia', 'nino', 'arbol', 'aire', 'natural', 'historia'],
    ['bano', 'limpio', 'faltar', 'cuidado', 'nino', 'prolijo', 'precio', 'comida', 'servicio', 'mantener'],
    ['familia', 'nino', 'tarde', 'tomar', 'libre', 'chico', 'aire', 'rato', 'edad', 'actividad'],
]
def sin_tildes(t):
    return ''.join(c for c in unicodedata.normalize('NFKD', str(t)) if unicodedata.category(c) != 'Mn')

res = pd.read_csv(f"{OUT}/LLM_results.csv")
nlp = spacy.load('es_core_news_sm', disable=['parser', 'ner'])
lem = [' '.join(t.lemma_ for t in d if t.is_alpha)
       for d in nlp.pipe(res['text'].astype(str).tolist(), batch_size=200)]
llm_docs = [set(sin_tildes(x).lower().split()) for x in lem]

tecnicas = [
    ("LDA",                lda_w, lda_docs, df_lda['tema_dominante'].to_numpy()),
    ("K-Means + TF-IDF",   tf_w,  tf_docs,  df_tf['cluster_tema'].to_numpy()),
    ("K-Means + Word2Vec", w2v_w, w2v_docs, df_w2v['cluster_w2v'].to_numpy()),
    ("BERTopic",           ber_w, ber_docs, df_ber['topico_bertopic'].to_numpy()),
    ("LLM (zero-shot)",    LLM_W, llm_docs, res['topic_id'].to_numpy()),
]

# =====================================================================
# 2. CHEQUEOS — si alguno falla, NO usar la figura
# =====================================================================
TD_ESPERADO = {"LDA": 0.9333, "K-Means + TF-IDF": 0.7000, "K-Means + Word2Vec": 1.0000,
               "BERTopic": 0.9333, "LLM (zero-shot)": 0.7000}
chk = {
    "BERTopic: ruido == 8.075 (valor de la tesis)":  int((df_ber['topico_bertopic'] == -1).sum()) == 8075,
    "BERTopic: ids alineados con df_generic_cleaned": bool((df_ber['id'].to_numpy() == base['id'].to_numpy()).all()),
    "LLM: 13 sin categoría":                          int((res['topic_id'] == -1).sum()) == 13,
}
for nombre, W, docs, lab in tecnicas:
    allw = [w for t in W for w in t]
    chk[f"{nombre}: textos alineados con asignaciones ({len(docs):,} = {len(lab):,})"] = len(docs) == len(lab)
    chk[f"{nombre}: 6 tópicos x 10 palabras"] = len(W) == 6 and all(len(t) == TOP_N for t in W)
    chk[f"{nombre}: TD == {TD_ESPERADO[nombre]:.4f}"] = abs(len(set(allw)) / len(allw) - TD_ESPERADO[nombre]) < 5e-5
for k, v in chk.items():
    print(f"  {'OK ' if v else 'FALLA'}  {k}")
assert all(chk.values()), "Algún chequeo falló: NO usar la figura. Avisar antes de seguir."

# =====================================================================
# 3. CÁLCULO: % de reseñas del tópico que contienen la palabra
#    + diagnóstico de correspondencia índice-tópico
# =====================================================================
PCT, filas = {}, []
print("\nDiagnóstico: ¿las palabras del tópico i son más frecuentes en el tópico i que en los otros?")
for nombre, W, docs, lab in tecnicas:
    idx = {t: np.where(lab == t)[0] for t in range(6)}
    def share(w, t):
        return sum(1 for i in idx[t] if w in docs[i]) / len(idx[t])
    P = np.array([[100 * share(w, t) for w in W[t]] for t in range(6)])
    S = np.array([[np.mean([share(w, j) for w in W[i]]) for j in range(6)] for i in range(6)])
    diag = int(sum(S[i].argmax() == i for i in range(6)))
    print(f"  {nombre:20} diagonal dominante en {diag}/6 tópicos")
    PCT[nombre] = P
    for t in range(6):
        for j, w in enumerate(W[t]):
            filas.append({'tecnica': nombre, 'topico': t, 'rango': j + 1, 'palabra': w,
                          'n_resenas_topico': len(idx[t]),
                          'resenas_con_palabra': int(round(P[t, j] / 100 * len(idx[t]))),
                          'pct': round(P[t, j], 2)})

pd.DataFrame(filas).to_csv(f"{OUT}/palabras_representativas_frecuencias.csv", index=False)

# =====================================================================
# 4. FIGURA (unidades = pulgadas; ancho = caja de texto de la tesis)
# =====================================================================
plt.rcParams.update({'font.family': 'sans-serif',
                     'font.sans-serif': ['Helvetica Neue', 'Helvetica', 'Arial', 'DejaVu Sans']})
AZUL  = '#3F5FBF'            # mismo azul que la Figura 4.5
W_FIG = 6.3
M, LBL_W, GAP = 0.05, 0.30, 0.06
CW    = (W_FIG - 2 * M - LBL_W - 5 * GAP) / 6
LINE, HEAD, PADB = 0.125, 0.20, 0.07
CH    = HEAD + TOP_N * LINE + PADB
RGAP, LEG = 0.12, 0.45
TIT    = 0.32                     # franja reservada para el título
TITULO = "Palabras representativas por técnica y tópico"
H_FIG  = M + TIT + 5 * CH + 4 * RGAP + LEG + M

fig = plt.figure(figsize=(W_FIG, H_FIG))
ax  = fig.add_axes([0, 0, 1, 1])
ax.set_xlim(0, W_FIG); ax.set_ylim(0, H_FIG); ax.axis('off')

ax.text(M + LBL_W + (W_FIG - 2 * M - LBL_W) / 2, H_FIG - M - TIT / 2, TITULO,
        ha='center', va='center', fontsize=9, fontweight='bold', color='0.15')

for r, (nombre, W, _, _) in enumerate(tecnicas):
    y_top = H_FIG - M - TIT - r * (CH + RGAP)
    y0    = y_top - CH
    ax.text(M + LBL_W / 2, y0 + CH / 2, nombre, rotation=90, ha='center', va='center',
            fontsize=7.2, fontweight='bold', color='0.15')
    for c in range(6):
        x0 = M + LBL_W + c * (CW + GAP)
        ax.add_patch(FancyBboxPatch((x0, y0), CW, CH, boxstyle="round,pad=0,rounding_size=0.035",
                                    fc='white', ec='0.78', lw=0.6))
        ax.text(x0 + 0.05, y_top - 0.055, f"T{c}", fontsize=6.4, fontweight='bold',
                color='0.25', va='top')
        for j, w in enumerate(W[c]):
            yc = y_top - HEAD - (j + 0.5) * LINE
            bx, bw, bh = x0 + 0.04, CW - 0.08, LINE * 0.78
            ax.add_patch(Rectangle((bx, yc - bh / 2), bw, bh, fc='0.94', ec='none'))
            ax.add_patch(Rectangle((bx, yc - bh / 2), bw * PCT[nombre][c, j] / 100, bh,
                                   fc=AZUL, alpha=0.56, ec='none'))
            ax.text(bx + 0.03, yc, w, fontsize=5.6, va='center', ha='left', color='0.1')

# Leyenda de escala
ly, lx, lw_, bh = M + 0.24, M + LBL_W, 1.4, LINE * 0.78
ax.add_patch(Rectangle((lx, ly - bh / 2), lw_, bh, fc='0.94', ec='none'))
ax.add_patch(Rectangle((lx, ly - bh / 2), lw_ * 0.5, bh, fc=AZUL, alpha=0.56, ec='none'))
for f, t in [(0, '0 %'), (0.5, '50 %'), (1, '100 %')]:
    ax.plot([lx + lw_ * f] * 2, [ly - bh / 2 - 0.03, ly - bh / 2], color='0.4', lw=0.5)
    ax.text(lx + lw_ * f, ly - bh / 2 - 0.05, t, fontsize=5, ha='center', va='top', color='0.35')
ax.text(lx + lw_ + 0.12, ly, "Porcentaje de las reseñas del tópico que contienen la palabra",
        fontsize=5.8, va='center', color='0.25')

fig.savefig(f"{OUT}/figura_palabras_representativas.png", dpi=300)
fig.savefig(f"{OUT}/figura_palabras_representativas.pdf")
plt.show()

### Figura 4.3

In [ ]:
# ============================================================
# UMAP COMPARATIVO — un unico espacio semantico, coloreado por cada modelo
# Reutiliza los embeddings ya cacheados (prep_bertopic_emb.npy). Calcula UN UMAP 2D y lo
# muestra en 6 paneles: densidad cruda + una particion por modelo (baseline semilla 42,
# tomada de los df_*_final). El espacio es NEUTRAL (embeddings), no el nativo de cada
# modelo -> se declara en el texto. W2V (~20k) y LLM (2000) cubren subconjuntos.
# ============================================================
import numpy as np, pandas as pd, os
import matplotlib.pyplot as plt
from umap import UMAP

BD    = ".."
DFDIR = f"{BD}/Topic/Dataframes"
RES   = f"{BD}/Topic/Resultados/Topico"

emb = np.load(f"{DFDIR}/prep_bertopic_emb.npy")
base = pd.read_pickle(f"{BD}/df_generic_cleaned.pkl").reset_index(drop=True)
assert len(base) == len(emb), f"desalineado: base={len(base)} vs emb={len(emb)} (re-correr el bloque BERTopic)"
ids = base['id'].values

U2 = f"{DFDIR}/prep_umap2d.npy"
if os.path.exists(U2):
    xy = np.load(U2); print("umap 2d cargado", xy.shape)
else:
    xy = UMAP(n_neighbors=15, min_dist=0.1, n_components=2, metric='cosine', random_state=42).fit_transform(emb)
    np.save(U2, xy); print("umap 2d creado", xy.shape)

dfxy = pd.DataFrame({'id': ids, 'x': xy[:, 0], 'y': xy[:, 1]})

def carga(path, col, nombre):
    d = pd.read_csv(path) if path.endswith('.csv') else pd.read_pickle(path)
    return d[['id', col]].rename(columns={col: nombre})

asig = {
    'LDA':      carga(f"{DFDIR}/df_lda_final.pkl", 'tema_dominante', 'LDA'),
    'TF-IDF':   carga(f"{DFDIR}/df_tfidf_final.pkl", 'cluster_tema', 'TF-IDF'),
    'W2V':      carga(f"{DFDIR}/df_w2v_final.pkl", 'cluster_w2v', 'W2V'),
    'BERTopic': carga(f"{DFDIR}/df_bertopic_final.pkl", 'topico_bertopic', 'BERTopic'),
    'LLM':      carga(f"{RES}/LLM_results_with_id.csv", 'topic_id', 'LLM'),
}

fig, axes = plt.subplots(3, 2, figsize=(13.3, 18)); axes = axes.flatten()
axes[0].scatter(dfxy['x'], dfxy['y'], s=2, c='lightgray', alpha=0.5)
axes[0].set_title(f'Espacio semántico (UMAP 2D, {len(dfxy)} reseñas)')
axes[0].set_xticks([]); axes[0].set_yticks([])
for ax, (nombre, a) in zip(axes[1:], asig.items()):
    m = dfxy.merge(a, on='id', how='left')
    cov = m[nombre].notna().mean() * 100
    ax.scatter(m['x'], m['y'], s=2, c='lightgray', alpha=0.25)      # no cubiertos
    sub = m[m[nombre].notna()]
    codes, _ = pd.factorize(sub[nombre])
    ax.scatter(sub['x'], sub['y'], s=2, c=codes, cmap='tab20', alpha=0.6)
    if nombre == 'BERTopic':
        ax.set_title('BERTopic (100% con etiqueta, incluido ruido)')
    else:
        ax.set_title(f'{nombre}  (cobertura {cov:.2f} %)'.replace('.', ','))
    ax.set_xticks([]); ax.set_yticks([])
plt.suptitle('Partición del espacio semántico por cada técnica (baseline semilla 42)', fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.97])      # reserva el 3 % superior para el título
plt.savefig(f"{RES}/UMAP_comparativo_modelos_3x2.png", dpi=300, bbox_inches='tight')
plt.show()
print("guardado -> UMAP_comparativo_modelos_3x2.png")

### Figura 4.4

In [ ]:
# ============================================================
# FIGURA 4.4 — Perfil temático por parque según LDA, corrida de referencia.
# Mapa de calor del Panel A de la Tabla 4.5: el porcentaje de reseñas de cada
# parque cuyo tópico dominante corresponde a cada uno de los seis tópicos.
# Es la figura que el texto lee "directamente", antes de la corrección que
# introduce la Figura 4.5.
#
# Se construye desde df_lda_final.pkl, la misma fuente que la Tabla 4.5 y la
# Figura 4.5, de modo que las tres coinciden por construcción.
# Autocontenido. No reentrena nada. Segundos.
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

def pc(v, d=2):
    """Formatea con coma decimal, como el resto del documento."""
    return f"{v:.{d}f}".replace(".", ",")

BD     = ".."
DFDIR  = f"{BD}/Topic/Dataframes"
SALIDA = f"{BD}/Topic/Resultados/Topico/figura_4_3_perfil_por_parque.png"
TITULO = "Porcentaje de reseñas de cada parque asignadas a cada tópico de LDA"

df_lda = pd.read_pickle(f"{DFDIR}/df_lda_final.pkl")

# --- Chequeos ---
print("CHEQUEOS")
ok = len(df_lda) == 27102
print(f"  filas de df_lda_final: {len(df_lda):,}          (esperado 27.102)")
tema = df_lda['tema_dominante'].to_numpy()
parque = df_lda['nombre_parque'].to_numpy()
print(f"  tópicos distintos: {sorted(set(tema))}          (esperado 0..5)")
ok &= sorted(set(tema)) == [0, 1, 2, 3, 4, 5]
print(f"  parques distintos: {len(set(parque))}                       (esperado 10)")
ok &= len(set(parque)) == 10
if not ok:
    raise SystemExit("\n>>> Algún chequeo falló. NO usar la figura.")
print("  todo OK\n")

perfil = pd.crosstab(parque, tema, normalize='index') * 100

orden = ['botanico', 'parque_prado', 'parque_rodo', 'rosedal-del_prado', 'villa_dolores',
         'jose_batlle_y_ordoniez', 'espacio_modelo', 'idea_vilarinio',
         'museo_y-parque_fernando_garcia', 'punta_yeguas']
etq = ['Jardín Botánico', 'Parque del Prado', 'Parque Rodó', 'Rosedal del Prado', 'Villa Dolores',
       'José Batlle y Ordóñez', 'Espacio Modelo', 'Idea Vilariño',
       'Museo y P. Fernando García', 'Punta Yeguas']
cols = ['T0\naire\nverde\nlibre',        'T1\ncaminar\ntarde\nárbol',
        'T2\nfamilia\nniño\njuego',    'T3\nnaturaleza\nfaltar\nbaño',
        'T4\ncuidado\nlimpio\nlago',   'T5\ntomar\nnatural\nprolijo']

M = perfil.reindex(orden).values

fig, ax = plt.subplots(figsize=(10, 5.6))
im = ax.imshow(M, cmap='Blues', vmin=0, vmax=M.max(), aspect='auto')

umbral = M.max() * 0.55
for i in range(M.shape[0]):
    for j in range(M.shape[1]):
        ax.text(j, i, pc(M[i, j]), ha='center', va='center', fontsize=8.5,
                color='white' if M[i, j] > umbral else '#22344F')

ax.set_xticks(range(len(cols))); ax.set_xticklabels(cols, fontsize=8, linespacing=1.35)
ax.set_yticks(range(len(etq)));  ax.set_yticklabels(etq, fontsize=9.5)
ax.tick_params(length=0)
ax.set_xticks(np.arange(-.5, len(cols), 1), minor=True)
ax.set_yticks(np.arange(-.5, len(etq), 1), minor=True)
ax.grid(which='minor', color='white', linewidth=2)
for s in ax.spines.values():
    s.set_visible(False)

cb = fig.colorbar(im, ax=ax, fraction=0.030, pad=0.02)
cb.set_label('% de las reseñas del parque', fontsize=8.5)
cb.outline.set_visible(False)
cb.ax.tick_params(length=0, labelsize=8)
cb.ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: pc(v, 0)))

ax.set_title(TITULO, fontsize=11, fontweight='bold', color='0.15', pad=12)

fig.tight_layout()
fig.savefig(SALIDA, dpi=300, bbox_inches='tight')
print(f"guardado -> {SALIDA}\n")

print("Panel A de la Tabla 4.5 (mismo objeto que la figura):")
print(perfil.reindex(orden).round(2).to_string())
print("\nTópico modal por parque:")
print(perfil.reindex(orden).idxmax(axis=1).to_string())

### Figura 4.5 y Tabla 4.5

In [ ]:
# ============================================================
# FIGURA 4.5 + TABLA 4.5 Panel B — T0 de LDA por parque con y sin las
# reseñas sin representación léxica. Dos decimales.
# Autocontenido: reconstruye la máscara de vacías desde cero. No reentrena nada.
# ============================================================

import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import CountVectorizer

def pc(v, d=2):
    """Formatea con coma decimal, como el resto del documento."""
    return f"{v:.{d}f}".replace(".", ",")

BD     = ".."
DFDIR  = f"{BD}/Topic/Dataframes"
RES    = f"{BD}/Topic/Resultados/Topico"
SALIDA = f"{RES}/figura_4_4_T0_por_parque.png"
TITULO = "Porcentaje de reseñas asignadas a T0 en cada parque"

# --- 1. Reconstruir la matriz documento-término, idéntica a la de LDA ---
prep = pickle.load(open(f"{DFDIR}/prep_lemmatizado.pkl", "rb"))
corpus, final_stopwords = prep['corpus'], prep['final_stopwords']

cv = CountVectorizer(stop_words=final_stopwords, min_df=10, max_df=0.6, ngram_range=(1, 2))
X = cv.fit_transform(corpus)
vac = np.asarray(X.getnnz(axis=1) == 0)

df_lda = pd.read_pickle(f"{DFDIR}/df_lda_final.pkl")

# --- 2. Chequeos de integridad ---
print("CHEQUEOS")
ok = True
print(f"  matriz {X.shape} · vocabulario {X.shape[1]} términos      (esperado 27102 x 1857)")
ok &= X.shape == (27102, 1857)
print(f"  filas vacías: {vac.sum():,} ({vac.mean():.2%})            (esperado 7.404 / 27,32 %)")
ok &= int(vac.sum()) == 7404
print(f"  filas de df_lda_final: {len(df_lda):,}                    (esperado 27.102)")
ok &= len(df_lda) == len(corpus)

tema = df_lda['tema_dominante'].to_numpy()
t0 = int((tema == 0).sum()); t0_vac = int(vac[tema == 0].sum())
print(f"  T0 = {t0:,} docs, de los cuales {t0_vac:,} vacías = {t0_vac/t0:.1%}   (esperado 11.258 / 7.404 / 65,8 %)")
ok &= (t0 == 11258) and (t0_vac == 7404)
print(f"  vacías fuera de T0: {int(vac.sum()) - t0_vac}                        (esperado 0)")
ok &= (int(vac.sum()) - t0_vac) == 0

if not ok:
    raise SystemExit("\n>>> Algún chequeo falló: el alineamiento no es el esperado. NO usar la figura.")
print("  todo OK\n")

# --- 3. Perfiles por parque ---
parque = df_lda['nombre_parque'].to_numpy()
perfil_todos  = pd.crosstab(parque, tema, normalize='index') * 100
perfil_limpio = pd.crosstab(parque[~vac], tema[~vac], normalize='index') * 100

orden = ['botanico', 'parque_prado', 'parque_rodo', 'rosedal-del_prado', 'villa_dolores',
         'jose_batlle_y_ordoniez', 'espacio_modelo', 'idea_vilarinio',
         'museo_y-parque_fernando_garcia', 'punta_yeguas']
etq = ['Jardín Botánico', 'Parque del Prado', 'Parque Rodó', 'Rosedal del Prado', 'Villa Dolores',
       'José Batlle y Ordóñez', 'Espacio Modelo', 'Idea Vilariño',
       'Museo y P. Fernando García', 'Punta Yeguas']

# --- 4. Tabla (Panel B) con dos decimales ---
s_parque  = pd.Series(parque)
n_con     = s_parque[~vac].value_counts().reindex(orden)
pct_sin   = pd.Series(vac).groupby(s_parque).mean().mul(100).reindex(orden)
PB        = perfil_limpio.reindex(orden)

assert int(n_con.sum()) == 27102 - 7404, "n con representación no suma 19.698"
assert np.allclose(PB.sum(axis=1), 100), "alguna fila del Panel B no suma 100"

tabla = pd.DataFrame({
    'Parque':        etq,
    'n con repr.':   [f"{v:,}".replace(",", ".") for v in n_con.values],
    '% sin repr.':   [pc(v) for v in pct_sin.values],
    **{f"T{t}": [pc(v) for v in PB[t].values] for t in range(6)},
})
print("Tabla 4.5, Panel B (sin las reseñas sin representación léxica):")
print(tabla.to_string(index=False))
tabla.to_csv(f"{RES}/tabla_4_5_panel_B_2dec.csv", index=False, sep=';', encoding='utf-8-sig')
print(f"\nguardado -> {RES}/tabla_4_5_panel_B_2dec.csv\n")

# --- 5. Figura 4.5 ---
a = perfil_todos.reindex(orden)[0].values
b = PB[0].values
x = np.arange(len(orden)); w = 0.38

fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(x - w/2, a, w, label='Todas las reseñas', color='#3F5FBF')
ax.bar(x + w/2, b, w, label='Excluyendo las reseñas sin representación léxica', color='#A9BCEA')
for xi, (va, vb) in enumerate(zip(a, b)):
    ax.text(xi - w/2, va + 0.8, pc(va), ha='center', fontsize=6.8)
    ax.text(xi + w/2, vb + 0.8, pc(vb), ha='center', fontsize=6.8)
ax.set_xticks(x); ax.set_xticklabels(etq, rotation=40, ha='right', fontsize=9)
ax.set_ylabel('% de reseñas del parque asignadas a T0')
ax.set_ylim(0, max(a) * 1.15)
ax.legend(frameon=False, fontsize=9)
ax.spines[['top', 'right']].set_visible(False)
ax.grid(axis='y', alpha=0.25, linewidth=0.6)
ax.set_axisbelow(True)
ax.set_title(TITULO, fontsize=11, fontweight='bold', color='0.15', pad=12)
fig.tight_layout()
fig.savefig(SALIDA, dpi=300, bbox_inches='tight')
print(f"guardado -> {SALIDA}")

# --- 6. Panel A, por si también lo querés con dos decimales ---
print("\nPanel A (todas):")
print(perfil_todos.reindex(orden).round(2).to_string())